# Template attack on Firmware Implementation of Kuznyechik (GOST)

## GOST Trace Capture

In [ ]:
SCOPETYPE = 'CWNANO' # options: OPENADC, CWNANO  
PLATFORM = 'CWNANO' # options: CWLITEXMEGA/CW308_XMEGA, CWLITEARM/CW308_STM32F3, CWNANO 
CRYPTO_TARGET='KUZNYECHIK'
SS_VER='SS_VER_1_1'

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
scope.adc.samples = 50000 #options: 96000 for CW-PRO (CW1200), 24400 for CW-Lite, 131070 for CW-Husky

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../firmware/mcu/simpleserial-gost
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3

In [ ]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-gost/simpleserial-gost-{}.hex".format(PLATFORM))

In [ ]:
from tqdm.notebook import trange
import numpy as np
import time
from os import urandom
from scipy.stats import multivariate_normal

trace_array = []
textin_array = []
keyin_array = []

text = urandom(16)

N = 100

for i in trange(N, desc='Capturing traces'):
    scope.arm()
    
    target.simpleserial_write('p', text)
    
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    
    response = target.simpleserial_read('r', 16)
    
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    
    text = urandom(16)
    
trace_array = np.array(trace_array)

In [ ]:
scope.dis()
target.dis()

In [ ]:
assert len(trace_array) == N
print("✔️ OK to continue!")

Again, let's quickly plot a trace to make sure everything looks as expected:

In [ ]:
%matplotlib notebook
import matplotlib.pylab as plt

plt.figure()
plt.plot(trace_array[0], 'r')
plt.plot(trace_array[N//2], 'g')
plt.plot(trace_array[N-1], 'b')
plt.show()

## GOST Model and Hamming Weight

In [ ]:
sbox = [
    252, 238, 221, 17, 207, 110, 49, 22, 251, 196, 250, 218, 35, 197, 4, 77, 233,
    119, 240, 219, 147, 46, 153, 186, 23, 54, 241, 187, 20, 205, 95, 193, 249, 24, 101,
    90, 226, 92, 239, 33, 129, 28, 60, 66, 139, 1, 142, 79, 5, 132, 2, 174, 227, 106, 143,
    160, 6, 11, 237, 152, 127, 212, 211, 31, 235, 52, 44, 81, 234, 200, 72, 171, 242, 42,
    104, 162, 253, 58, 206, 204, 181, 112, 14, 86, 8, 12, 118, 18, 191, 114, 19, 71, 156,
    183, 93, 135, 21, 161, 150, 41, 16, 123, 154, 199, 243, 145, 120, 111, 157, 158, 178,
    177, 50, 117, 25, 61, 255, 53, 138, 126, 109, 84, 198, 128, 195, 189, 13, 87, 223,
    245, 36, 169, 62, 168, 67, 201, 215, 121, 214, 246, 124, 34, 185, 3, 224, 15, 236,
    222, 122, 148, 176, 188, 220, 232, 40, 80, 78, 51, 10, 74, 167, 151, 96, 115, 30, 0,
    98, 68, 26, 184, 56, 130, 100, 159, 38, 65, 173, 69, 70, 146, 39, 94, 85, 47, 140, 163,
    165, 125, 105, 213, 149, 59, 7, 88, 179, 64, 134, 172, 29, 247, 48, 55, 107, 228, 136,
    217, 231, 137, 225, 27, 131, 73, 76, 63, 248, 254, 141, 83, 170, 144, 202, 216, 133,
    97, 32, 113, 103, 164, 45, 43, 9, 91, 203, 155, 37, 208, 190, 229, 108, 82, 89, 166,
    116, 210, 230, 244, 180, 192, 209, 102, 175, 194, 57, 75, 99, 182
]

kuz_lvec = [0x94, 0x20, 0x85, 0x10, 0xC2, 0xC0, 0x01, 0xFB,
            0x01, 0xC0, 0xC2, 0x10, 0x85, 0x20, 0x94, 0x01]

def kuz_mul_gf256(x, y):
    z = 0
    for i in range(8):
        if (y & 1):
            z ^= x
        if (x & 0x80):
            x = ((x << 1) & 0xFF) ^ 0xC3
        else:
            x = (x << 1) & 0xFF
        y >>= 1
    return z

def L(inputdata):
    res = [block for block in inputdata]
    x = 0
    for j in range(16):
        # An LFSR with 16 elements from GF(2^8)
        x = res[15] # since lvec[15] = 1

        for i in range(14,-1,-1):
            res[i+1] = res[i]
            x ^= kuz_mul_gf256(res[i], kuz_lvec[i])
        res[0] = x
    return res

def SX(inputdata, key):
    return sbox[inputdata ^ key]

HW = [bin(n).count("1") for n in range(0, 256)]

## Developing our Template algorithm

To distinguish the operation we need from the noise in the measurements, we need to use probability density function (PDF):
$$f(x) = \frac{1}{\sqrt{(2\pi)^k{|\sum|}}}{e}^{{-((x - \mu)^T}{\sum^{-1}}{(x - \mu)/2)}}$$

To work with multiple values that may depend on each other, a covariance matrix is used:
$$\sum = \begin{bmatrix}
cov(x, x) & cov(x, y) & cov(x, z) & ... \\
cov(y, x) & cov(y, y) & cov(y, z) & ... \\
cov(z, x) & cov(z, y) & cov(z, z) & ... \\
... & ... & ... & ... \\
\end{bmatrix}
$$

Mean of each random variable:
$$\mu = \begin{bmatrix}
\mu_x \\
\mu_y \\
\mu_z \\
... \\
\end{bmatrix}
$$

Not every point on the power trace is important to us, so we can use 3-5 points of interest (POI). To find these points we need to find the average power $M_{k,j}$ for each operation $k$ and sample $i$:
$$M_{k,i} = \frac{1}{T_k}{\sum^{T_k}_{j=1}{t_{j,i}}}$$

Then we need to find their absolute pairwise difference. Find the largest value, throw out the closest $N$ points, repeat until enough POIs have been selected:
$$D_i = \sum_{k_1,k_2}{|M_{k_1,i} - M_{k_2,i}|}$$

Put trace values at the POIs into a vector:
$$a_j = \begin{bmatrix}
a_{j,1} \\
a_{j,2} \\
a_{j,3} \\
... \\
\end{bmatrix}
$$

Calculate the PDF for every key guess:
$$p_{k,j} = f_k(a_j)$$

Combine $p_{k,j}$ values:
$$\log{P_k} = \sum_{j=1}^{A}{\log{p_{k,j}}}$$

## Template Attack Implementaiton

### Creating the template

#### Sort traces:

In [ ]:
temp_sbox = [L(SX(textin_array[i], keyin_array[i])) for i in range(len(textin_array) - 1)]
temp_hw = [hw[s] for s in temp_sbox]

# 2.5: Sort traces by HW
# Make 9 blank lists - one for each Hamming weight
temp_traces_hw = [[] for _ in range(9)]

# Fill them up
for i in range(len(trace_array) - 1):
    HW = temp_hw[i]
    temp_traces_hw[HW].append(trace_array[i])

# Switch to numpy arrays
temp_traces_hw = [np.array(temp_traces_hw[HW]) for HW in range(9)]

#### Finding points of interest:

In [ ]:
# 3: Find averages
temp_means = np.zeros((9, len(trace_array[0])))
for i in range(9):
    temp_means[i] = np.average(temp_traces_hw[i], 0)
    
# 4: Find sum of differences
temp_sum_diff = np.zeros(len(trace_array[0]))
for i in range(9):
    for j in range(i):
        temp_sum_diff += np.abs(temp_means[i] - temp_means[j])

# 5: Find POIs
POIs = []
numPOIs = 5
POIspacing = 5
for i in range(numPOIs):
    # Find the max
    nextPOI = temp_sum_diff.argmax()
    POIs.append(nextPOI)

    # Make sure we don't pick a nearby value
    poi_min = max(0, nextPOI - POIspacing)
    poi_max = min(nextPOI + POIspacing, len(temp_sum_diff))
    for j in range(poi_min, poi_max):
        temp_sum_diff[j] = 0

#### Covariance matrix:

In [ ]:
# 6: Fill up mean and covariance matrix for each HW
mean_matrix = np.zeros((9, numPOIs))
cov_matrix = np.zeros((9, numPOIs, numPOIs))
for HW in range(9):
    for i in range(numPOIs):
        # Fill in mean
        mean_matrix[HW][i] = temp_means[HW][POIs[i]]
        for j in range(numPOIs):
            x = temp_traces_hw[HW][:, POIs[i]]
            y = temp_traces_hw[HW][:, POIs[j]]
            cov_matrix[HW, i, j] = np.cov(x, y)[0][1]

### Perfoming the attack

In [ ]:
# 1: Load attack traces
atk_traces = trace_array[-1]
atk_ptext = textin_array[-1]
atk_key = keyin_array[-1]

print(atk_key)

# 2: Attack
# Running total of log P_k
p_k = np.zeros(256)
for j in range(len(atk_traces)):
    # Grab key points and put them in a small matrix
    a = [atk_traces[j][POIs[i]] for i in range(len(POIs))]

    # Test each key
    for k in range(256):
        # Find HW coming out of sbox
        HW = hw[sbox[atk_ptext[j][0] ^ k]]

        # Find p_{k,j}
        rv = multivariate_normal(mean_matrix[HW], cov_matrix[HW])
        p_kj = rv.pdf(a)

        # Add it to running total
        p_k[k] += np.log(p_kj)

    # Print our top 5 results so far
    # Best match on the right
    print(p_k.argsort()[-5:])